# 01 — Single Product Harness

This notebook is the **single-product gateway** for the Product Evidence Harness. It proves one product end to end and shows the default reviewer-facing artifacts.

```mermaid
flowchart LR
    A[Product Input] --> B[Search]
    B --> C[Candidate Tournament]
    C --> D[Champion URL]
    D --> E[Production URL Gate]
    E --> F[Concise Review Packet]
    F --> G[Product Coding Input]
```

## Default row packet

```text
output/<row_id>/
├── final_row.csv
├── review_summary.md
├── review_decision.json
├── candidate_decisions.csv
└── product_coding_input.json
```

Deep debug artifacts such as tournament brackets, full traces, and verbose markdown are optional diagnostics only. The default business review path is the concise packet above.


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"

# Keep both paths because the package still has internal src.product_evidence_harness imports.
for path in (PROJECT_ROOT, SRC_PATH):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

PROJECT_ROOT


In [ ]:
from product_evidence_harness import (
    HarnessConfig,
    ProductEvidenceHarness,
    ProductQuery,
    ProductionURLGate,
    SerpAPIConfig,
    configure_logging,
)

configure_logging("INFO")
print("import ok")


## 1. Product input

Required: `main_text`, `country_code`. Optional: `row_id`, `ean`, `retailer_name`, `language_code`, `region`. Keep EAN/GTIN as text.


In [ ]:
product = ProductQuery(
    row_id="demo-single-001",
    main_text="PUT PRODUCT TEXT HERE",
    country_code="CZ",
    ean="",
    retailer_name="",
)

product


## 2. Configuration

Tournament mode is the primary discovery path. The concise review packet is the primary reviewer-facing output.


In [ ]:
config = HarnessConfig.from_env(PROJECT_ROOT / ".env")
tournament = getattr(config, "tournament", None)

pd.Series({
    "output_dir": str(getattr(config, "output_dir", "output")),
    "tournament_enabled": getattr(tournament, "enabled", None),
    "max_serp_credits": getattr(tournament, "max_serp_credits", None),
    "candidate_pool": getattr(tournament, "candidate_pool", None),
    "preflight_top_k": getattr(tournament, "preflight_top_k", None),
    "batch_size": getattr(tournament, "batch_size", None),
    "max_batches": getattr(tournament, "max_batches", None),
    "write_review_pack": getattr(config, "write_review_pack", "default/unknown"),
    "write_markdown_reports": getattr(config, "write_markdown_reports", "default/unknown"),
    "write_trace_json": getattr(config, "write_trace_json", "default/unknown"),
    "write_debug_csvs": getattr(config, "write_debug_csvs", "default/unknown"),
}).to_frame("value")


## 3. Run the harness

This executes the full evidence flow for one product.


In [ ]:
serp_config = SerpAPIConfig.from_env(
    country_code=product.country_code,
    language_code=product.language_code or "en",
    env_file=PROJECT_ROOT / ".env",
)

harness = ProductEvidenceHarness(serp_config=serp_config, config=config)
trace = harness.run(product, return_trace=True)
match = trace.best_match
match


## 4. Decision snapshot

A URL is automated handoff-ready only when the production gate and champion confirmation both pass.


In [ ]:
tournament_result = getattr(trace.state, "tournament_result", None)
confirmation = getattr(tournament_result, "champion_confirmation", None) if tournament_result else None
production = ProductionURLGate().assess_url_in_state(trace.state, match.product_url or "")

decision_summary = {
    "product_url": getattr(match, "product_url", None),
    "verified_exact_url": getattr(match, "verified_exact_url", None),
    "confidence": getattr(match, "confidence", None),
    "needs_review": getattr(match, "needs_review", None),
    "url_decision_status": getattr(match, "url_decision_status", None),
    "production_url_ready": getattr(production, "production_ready", False) if production else False,
    "production_url_status": getattr(production, "status", "NO_PRODUCTION_ASSESSMENT") if production else "NO_PRODUCTION_ASSESSMENT",
    "browser_openable": getattr(production, "browser_openable", False) if production else False,
    "highly_scrapable": getattr(production, "highly_scrapable", False) if production else False,
    "exact_product_url_match": getattr(production, "exact_product_match", False) if production else False,
    "champion_confirmation_passed": getattr(confirmation, "passed", False) if confirmation else False,
    "champion_confirmation_status": getattr(confirmation, "status", "NO_CONFIRMATION") if confirmation else "NO_CONFIRMATION",
    "champion_confirmation_success_count": getattr(confirmation, "success_count", 0) if confirmation else 0,
    "champion_confirmation_required_successes": getattr(confirmation, "required_successes", 3) if confirmation else 3,
    "tournament_champion_url": getattr(tournament_result, "champion_url", None) if tournament_result else None,
    "tournament_runner_up_url": getattr(tournament_result, "runner_up_url", None) if tournament_result else None,
    "tournament_champion_margin": getattr(tournament_result, "champion_margin", None) if tournament_result else None,
}
pd.Series(decision_summary).to_frame("value")


In [ ]:
handoff_ready = bool(
    production
    and getattr(production, "production_ready", False)
    and getattr(production, "status", "") == "PRODUCTION_READY_EXACT_SCRAPABLE_BROWSER_URL"
    and confirmation
    and getattr(confirmation, "passed", False)
    and getattr(confirmation, "success_count", 0) >= getattr(confirmation, "required_successes", 3)
    and not getattr(match, "needs_review", True)
)

print("HANDOFF_READY_FOR_BROWSER_SCRAPING_AND_PRODUCT_CODING =", handoff_ready)
if not handoff_ready:
    print("This URL may still be useful, but it is review-only until all gates pass.")


## 5. Inspect concise row packet

Open `review_summary.md` first. Use `04_review_artifact_reader.ipynb` to render the row decision in notebook form.


In [ ]:
output_dir = Path(getattr(config, "output_dir", PROJECT_ROOT / "output"))
row_dir = output_dir / product.row_id
expected = ["final_row.csv", "review_summary.md", "review_decision.json", "candidate_decisions.csv", "product_coding_input.json"]

print("Row artifact directory:", row_dir)
if row_dir.exists():
    existing = {p.name for p in row_dir.iterdir()}
    display(pd.DataFrame({"artifact": expected, "exists": [name in existing for name in expected]}))
else:
    print("Row directory not found. Check PRODUCT_HARNESS_OUTPUT_DIR / config.output_dir.")


In [ ]:
summary_path = row_dir / "review_summary.md"
decision_path = row_dir / "review_decision.json"
candidate_path = row_dir / "candidate_decisions.csv"

if summary_path.exists():
    print(summary_path)
    print("\nFirst 1600 characters:\n")
    print(summary_path.read_text(encoding="utf-8")[:1600])
else:
    print("review_summary.md not found")


In [ ]:
if decision_path.exists():
    review_decision = json.loads(decision_path.read_text(encoding="utf-8"))
    display(pd.DataFrame([review_decision.get("decision", {})]).T.rename(columns={0: "value"}))
    display(pd.DataFrame([review_decision.get("checks", {})]).T.rename(columns={0: "value"}))
else:
    print("review_decision.json not found")


In [ ]:
if candidate_path.exists():
    candidates = pd.read_csv(candidate_path, dtype=str)
    review_columns = [
        "rank", "selected", "decision", "url", "confidence", "validation_status",
        "identity_status", "exact_product_check", "country_check", "retailer_check",
        "scrapable", "product_page", "reason",
    ]
    display(candidates[[c for c in review_columns if c in candidates.columns]].head(25))
else:
    print("candidate_decisions.csv not found")


## 6. Optional diagnostics

Deep artifacts are optional diagnostics. Use them only for debugging or audit escalation.


In [ ]:
optional_debug_artifacts = [
    "tournament_bracket.md", "tournament_bracket.json",
    "champion_confirmation.md", "champion_confirmation.json",
    "batch_winners.csv", "quality_assessment.md", "evidence_graph.json", "trace.json",
]
if row_dir.exists():
    existing = {p.name for p in row_dir.iterdir()}
    display(pd.DataFrame({
        "optional_debug_artifact": optional_debug_artifacts,
        "exists": [name in existing for name in optional_debug_artifacts],
    }))
